# Treino v3 — Violência contra a Mulher (SINAN)

Este notebook gera os **6 modelos** usados pelo app Streamlit (2 algoritmos × 3 cenários). Cada um tem o **seu próprio conjunto de variáveis**, escolhido por seleção automática.

| Cenário | Alvo | Pergunta |
|---|---|---|
| Ciclo de Reincidência | `target_reincidencia` | A violência já tinha ocorrido outras vezes? |
| Agressor Íntimo | `autor_parceiro_ou_ex` | O agressor é parceiro ou ex? |
| Encaminhamento DEAM | `encaminhamento_delegacia_mulher` | O caso foi encaminhado à Delegacia da Mulher? |

**O que mudou em relação ao `export_multi_models_v2.py`**

1. **Seleção de variáveis por modelo e por cenário.** Usa eliminação recursiva guiada por importância de permutação, medida numa validação interna. Fica o menor subconjunto cuja AUC esteja a até 1 erro-padrão da melhor (regra de 1 EP).
2. **Mais candidatas.** São as 26 variáveis da v2, a hora, o mês, o local completo, o vínculo do agressor e 9 variáveis derivadas, num total de 41. O vínculo fica de fora no cenário "Agressor Íntimo", porque ali ele é o próprio alvo. Os encaminhamentos ficam de fora em todos os cenários, porque são decisões tomadas no mesmo atendimento (vazamento).
3. **"Ignorado" é tratado como categoria.** Hora, local e município não informados não são mais contados como "não".
4. **Probabilidades calibradas.** O modelo é treinado sem `class_weight`, e o limiar de decisão é escolhido na validação pelo índice de Youden. O app permite mover esse limiar.
5. **Comparação justa com a v2.** A configuração da v2 é retreinada nos **mesmos dados e na mesma divisão** e entra no `metrics.json` como baseline.
6. **Validação temporal.** O treino usa casos até 2021 e o teste usa 2022–2025, para medir quanto o modelo aguenta mudanças ao longo do tempo.

**Como usar:** ajuste a célula de **Configuração** e rode tudo (*Ambiente de execução → Executar tudo*). No final é baixado um `modelos_v3.zip` com tudo que o app precisa. Com `MODO_RAPIDO = False` o treino completo leva cerca de **40–60 min** no Colab gratuito. Com `True` leva ~3 min e serve só para testar.

In [ ]:
import sys, os, gc, json, time, platform, datetime as dt, importlib, shutil, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn

IN_COLAB = "google.colab" in sys.modules
warnings.filterwarnings("ignore", message="Unknown solver options")  # aviso inofensivo do scipy com a LR
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | Colab: {IN_COLAB}")
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 2), "Precisa de scikit-learn >= 1.2"

## 1. Configuração

* **`CSV_PATH`**: caminho do CSV exportado pela `query_violencia_v3.sql`. Pode ser `.csv` ou `.zip`. Se você exportou para o Google Drive, deixe `USAR_DRIVE = True` e aponte para `/content/drive/MyDrive/...`. Se preferir enviar o arquivo pelo painel lateral do Colab, use `USAR_DRIVE = False` e o caminho `/content/<arquivo>`.
* **`MODO_RAPIDO`**: usa amostras pequenas, só para conferir se tudo roda.
* **`COPIAR_PARA_DRIVE`**: além do download, salva o `.zip` final numa pasta do Drive.

In [ ]:
CSV_PATH = "/content/drive/MyDrive/bq-results-20261105-010847-1793840970985.csv"
USAR_DRIVE = True
COPIAR_PARA_DRIVE = True
PASTA_DRIVE_SAIDA = "/content/drive/MyDrive/modelos_violencia_v3"
PASTA_SAIDA = "saida_modelos_v3"
MODO_RAPIDO = False

# Cenários a rodar (útil para retomar se a sessão cair: os já concluídos ficam salvos em PASTA_SAIDA)
CENARIOS = ["target_reincidencia", "autor_parceiro_ou_ex", "encaminhamento_delegacia_mulher"]

CFG = dict(
    seed=42,
    sel_fit=120_000,         # linhas para ajustar os modelos durante a seleção
    sel_val=40_000,          # linhas de validação durante a seleção
    perm_repeats=3,          # repetições da importância por permutação
    tol=None,                # tolerância de AUC na escolha do subconjunto (None = 1 erro-padrão)
    gbdt_select_iter=120,
    gbdt_final_iter=1500,     # teto; o early stopping para antes
    max_train=None,          # limite de linhas no ajuste final (None = todas)
    temporal_corte=2021,     # treino <= corte, teste > corte
    temporal_max_train=600_000,
)
if MODO_RAPIDO:
    CFG.update(sel_fit=15_000, sel_val=8_000, perm_repeats=2, gbdt_select_iter=40, gbdt_final_iter=120,
               max_train=40_000, temporal_max_train=40_000)

# Permite sobrescrever por variável de ambiente (útil fora do Colab)
CSV_PATH = os.environ.get("CSV_PATH", CSV_PATH)
PASTA_SAIDA = os.environ.get("PASTA_SAIDA", PASTA_SAIDA)
if os.environ.get("MODO_RAPIDO"):
    MODO_RAPIDO = os.environ["MODO_RAPIDO"] == "1"
    if MODO_RAPIDO:
        CFG.update(sel_fit=15_000, sel_val=8_000, perm_repeats=2, gbdt_select_iter=40, gbdt_final_iter=120,
                   max_train=40_000, temporal_max_train=40_000)
print("MODO_RAPIDO =", MODO_RAPIDO)
print(json.dumps(CFG, indent=1))

## 2. Módulo `features.py`

A célula abaixo grava o `features.py`, que concentra as colunas, a normalização de códigos e as variáveis derivadas. Ele precisa ir junto com o app: os modelos salvos chamam `features.prepare_features` na primeira etapa do pipeline. Assim o app envia a linha crua do formulário e o pipeline faz o resto. É isso que impede de voltar o bug da v2, em que o app mandava `"4"` e o modelo esperava `"4.0"`.

In [ ]:
%%writefile features.py
# features.py
"""
Esquema de variáveis compartilhado entre o treino (notebook treino_modelos_v3.ipynb) e o app (app.py).

Concentra num só lugar:
  * as colunas brutas do CSV exportado do BigQuery (query_violencia_v3.sql);
  * a normalização dos códigos categóricos (4 / "4" / "4.0" -> "4"; nulo -> "ignorado");
  * as variáveis derivadas (hora -> noite/faixa horária, contagens, faixa etária, região...);
  * os rótulos usados na interface.

IMPORTANTE: os pipelines salvos em .joblib começam com
`FunctionTransformer(features.prepare_features)`. Por isso este arquivo precisa estar
no mesmo diretório do app.py. O app envia a linha "crua" e o próprio pipeline cuida
de tipagem, normalização e variáveis derivadas — treino e inferência nunca divergem.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

MISSING = "ignorado"

# ---------------------------------------------------------------------------
# 1. Colunas brutas (entradas do formulário do app)
# ---------------------------------------------------------------------------
RAW_NUM_COLS = ["idade_paciente", "numero_envolvidos", "hora_ocorrencia_h"]
RAW_CAT_COLS = [
    "dia_semana_ocorrencia", "mes_ocorrencia", "uf_ocorrencia", "local_ocorrencia_cod",
    "reside_municipio_ocorrencia", "raca_paciente", "escolaridade_paciente",
    "estado_civil_paciente", "autor_sexo",
]
VIOLENCIA_COLS = [
    "violencia_fisica", "violencia_psicologica", "violencia_sexual",
    "violencia_financeira", "violencia_negligencia", "violencia_tortura",
]
MEIO_COLS = [
    "meio_forca_corporal", "meio_enforcamento", "meio_objeto_contundente",
    "meio_objeto_perfurante", "meio_arma_fogo", "meio_ameaca",
]
# Vínculo do agressor. Fica fora do cenário "Agressor Íntimo" (o vínculo é o próprio alvo).
VINCULO_COLS = ["autor_parceiro_ou_ex", "autor_familiar", "autor_conhecido", "autor_desconhecido"]
RAW_BIN_COLS = [
    "gestante", "possui_deficiencia", *VIOLENCIA_COLS, *MEIO_COLS,
    "autor_alcoolizado", *VINCULO_COLS, "ocorrencia_antiga",
]
RAW_COLS = RAW_NUM_COLS + RAW_CAT_COLS + RAW_BIN_COLS

TARGETS = {
    "Ciclo de Reincidência": "target_reincidencia",
    "Agressor Íntimo (Parceiro/Ex)": "autor_parceiro_ou_ex",
    "Encaminhamento DEAM": "encaminhamento_delegacia_mulher",
}
# Colunas do CSV que nunca entram como variável (vazamento ou identificação)
NEVER_FEATURES = [
    "encaminhamento_assistencia_social", "encaminhamento_atendimento_mulher",
    "encaminhamento_delegacia_mulher", "target_reincidencia",
    "ano", "data_ocorrencia", "id_municipio_ocorrencia", "nome_municipio_ocorrencia",
    "hora_ocorrencia_raw",
]

# ---------------------------------------------------------------------------
# 2. Variáveis derivadas (calculadas a partir das brutas, sem novos campos)
# ---------------------------------------------------------------------------
UF_REGIAO = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}
FAIXAS_ETARIAS = [(0, 17, "14-17"), (18, 24, "18-24"), (25, 34, "25-34"),
                  (35, 44, "35-44"), (45, 59, "45-59"), (60, 200, "60+")]
FAIXAS_HORARIAS = [(0, 5, "madrugada"), (6, 11, "manha"), (12, 17, "tarde"), (18, 23, "noite")]
LETAL_COLS = ["meio_arma_fogo", "meio_objeto_perfurante", "meio_enforcamento"]

DERIVED = {
    "ocorreu_noite_madrugada": {"kind": "cat", "sources": ["hora_ocorrencia_h"]},
    "faixa_horaria": {"kind": "cat", "sources": ["hora_ocorrencia_h"]},
    "local_residencia": {"kind": "cat", "sources": ["local_ocorrencia_cod"]},
    "n_tipos_violencia": {"kind": "num", "sources": VIOLENCIA_COLS},
    "n_meios_agressao": {"kind": "num", "sources": MEIO_COLS},
    "meio_potencialmente_letal": {"kind": "bin", "sources": LETAL_COLS},
    "faixa_etaria": {"kind": "cat", "sources": ["idade_paciente"]},
    "fim_de_semana": {"kind": "bin", "sources": ["dia_semana_ocorrencia"]},
    "regiao_ocorrencia": {"kind": "cat", "sources": ["uf_ocorrencia"]},
}
DERIVED_COLS = list(DERIVED)
ALL_CANDIDATES = RAW_COLS + DERIVED_COLS


def kind_of(col: str) -> str:
    """'num', 'cat' ou 'bin'."""
    if col in RAW_NUM_COLS:
        return "num"
    if col in RAW_CAT_COLS:
        return "cat"
    if col in RAW_BIN_COLS:
        return "bin"
    if col in DERIVED:
        return DERIVED[col]["kind"]
    raise KeyError(col)


def sources_of(col: str) -> list[str]:
    """Campos brutos de que uma variável depende (ela mesma, se for bruta)."""
    return list(DERIVED[col]["sources"]) if col in DERIVED else [col]


def raw_inputs_for(features: list[str]) -> list[str]:
    """Campos brutos (do formulário) necessários para um conjunto de variáveis."""
    used = {src for f in features for src in sources_of(f)}
    return [c for c in RAW_COLS if c in used]


# ---------------------------------------------------------------------------
# 3. Normalização e preparo
# ---------------------------------------------------------------------------
def normalize_code(value, missing: str = MISSING) -> str:
    """4 / '4' / '4.0' / ' 4 ' -> '4';  NaN / '' / None -> 'ignorado'."""
    if value is None:
        return missing
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return missing
    s = str(value).strip()
    if s == "" or s.lower() in {"nan", "none", "null", "<na>", missing}:
        return missing
    try:
        f = float(s.replace(",", "."))
        if np.isfinite(f) and f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s


def _normalize_series(ser: pd.Series, col: str) -> pd.Series:
    mapping = {}
    for u in pd.unique(ser):
        code = normalize_code(u)
        if code != MISSING:
            if col == "uf_ocorrencia":
                code = code.upper()
            elif col == "autor_sexo":
                code = {"M": "M", "1": "M", "F": "F", "2": "F", "AMBOS": "Ambos", "3": "Ambos"}.get(code.upper(), MISSING)
            elif col == "reside_municipio_ocorrencia":
                code = code if code in ("0", "1") else MISSING
        mapping[u] = code

    def get(v):
        if v is None or (isinstance(v, (float, np.floating)) and np.isnan(v)):
            return MISSING
        return mapping.get(v, MISSING)

    return pd.Series([get(v) for v in ser.to_numpy(dtype=object)], index=ser.index, dtype=object)


def _bins(values: pd.Series, bins) -> pd.Series:
    out = np.full(len(values), MISSING, dtype=object)
    v = values.to_numpy(dtype=float)
    for lo, hi, lab in bins:
        out[(v >= lo) & (v <= hi)] = lab
    return pd.Series(out, index=values.index, dtype=object)


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tipagem + normalização + variáveis derivadas. Idempotente.

    Recebe um DataFrame com (pelo menos parte das) colunas brutas e devolve um DataFrame
    com todas as ALL_CANDIDATES. Colunas ausentes viram faltantes (numéricas -> NaN,
    categóricas -> 'ignorado', binárias -> 0).
    """
    df = pd.DataFrame(df)
    out = pd.DataFrame(index=df.index)

    for c in RAW_NUM_COLS:
        out[c] = pd.to_numeric(df[c], errors="coerce").astype(float) if c in df else np.nan
    out.loc[(out["idade_paciente"] < 0) | (out["idade_paciente"] > 110), "idade_paciente"] = np.nan
    out.loc[~out["numero_envolvidos"].isin([1, 2]), "numero_envolvidos"] = np.nan   # 9 = ignorado
    out.loc[(out["hora_ocorrencia_h"] < 0) | (out["hora_ocorrencia_h"] > 23), "hora_ocorrencia_h"] = np.nan

    for c in RAW_CAT_COLS:
        out[c] = _normalize_series(df[c], c) if c in df else MISSING

    for c in RAW_BIN_COLS:
        # Qualquer valor diferente de 1 (0, 2 = "não" no SINAN, 9 = ignorado, NaN) vira 0
        out[c] = (pd.to_numeric(df[c], errors="coerce") == 1).astype(int) if c in df else 0

    # --- derivadas ---
    hora = out["hora_ocorrencia_h"]
    if "hora_ocorrencia_h" in df or "ocorreu_noite_madrugada" not in df:
        noite = np.where(hora.isna(), MISSING, np.where((hora >= 18) | (hora < 6), "1", "0"))
        out["ocorreu_noite_madrugada"] = pd.Series(noite, index=out.index, dtype=object)
    else:  # compatibilidade com CSVs antigos, que só tinham a coluna binária
        out["ocorreu_noite_madrugada"] = _normalize_series(df["ocorreu_noite_madrugada"], "reside_municipio_ocorrencia")
    out["faixa_horaria"] = _bins(hora, FAIXAS_HORARIAS)

    if "local_ocorrencia_cod" in df or "local_residencia" not in df:
        loc = out["local_ocorrencia_cod"]
        out["local_residencia"] = pd.Series(np.where(loc == MISSING, MISSING, np.where(loc == "1", "1", "0")),
                                            index=out.index, dtype=object)
    else:
        out["local_residencia"] = _normalize_series(df["local_residencia"], "reside_municipio_ocorrencia")

    out["n_tipos_violencia"] = out[VIOLENCIA_COLS].sum(axis=1).astype(float)
    out["n_meios_agressao"] = out[MEIO_COLS].sum(axis=1).astype(float)
    out["meio_potencialmente_letal"] = (out[LETAL_COLS].sum(axis=1) > 0).astype(int)
    out["faixa_etaria"] = _bins(out["idade_paciente"], FAIXAS_ETARIAS)
    out["fim_de_semana"] = out["dia_semana_ocorrencia"].isin(["1", "7"]).astype(int)
    out["regiao_ocorrencia"] = out["uf_ocorrencia"].map(UF_REGIAO).fillna(MISSING).astype(object)

    return out[ALL_CANDIDATES]


# ---------------------------------------------------------------------------
# 4. Rótulos para a interface
# ---------------------------------------------------------------------------
LABELS = {
    "idade_paciente": "Idade da vítima",
    "numero_envolvidos": "Nº de agressores",
    "hora_ocorrencia_h": "Hora da ocorrência",
    "dia_semana_ocorrencia": "Dia da semana",
    "mes_ocorrencia": "Mês da ocorrência",
    "uf_ocorrencia": "UF da ocorrência",
    "local_ocorrencia_cod": "Local da ocorrência",
    "reside_municipio_ocorrencia": "Mora no município do fato",
    "raca_paciente": "Raça/cor da vítima",
    "escolaridade_paciente": "Escolaridade da vítima",
    "estado_civil_paciente": "Situação conjugal da vítima",
    "autor_sexo": "Sexo do agressor",
    "gestante": "Vítima gestante",
    "possui_deficiencia": "Vítima com deficiência/transtorno",
    "violencia_fisica": "Violência física",
    "violencia_psicologica": "Violência psicológica",
    "violencia_sexual": "Violência sexual",
    "violencia_financeira": "Violência financeira/patrimonial",
    "violencia_negligencia": "Negligência/abandono",
    "violencia_tortura": "Tortura",
    "meio_forca_corporal": "Força corporal/espancamento",
    "meio_enforcamento": "Enforcamento/asfixia",
    "meio_objeto_contundente": "Objeto contundente",
    "meio_objeto_perfurante": "Objeto perfurocortante",
    "meio_arma_fogo": "Arma de fogo",
    "meio_ameaca": "Ameaça",
    "autor_alcoolizado": "Agressor sob efeito de álcool",
    "autor_parceiro_ou_ex": "Agressor é parceiro/ex",
    "autor_familiar": "Agressor é familiar",
    "autor_conhecido": "Agressor é conhecido",
    "autor_desconhecido": "Agressor é desconhecido",
    "ocorrencia_antiga": "Fato com mais de 1 ano",
    "ocorreu_noite_madrugada": "Noite/madrugada (18h–6h)",
    "faixa_horaria": "Faixa horária",
    "local_residencia": "Ocorreu na residência",
    "n_tipos_violencia": "Nº de tipos de violência",
    "n_meios_agressao": "Nº de meios de agressão",
    "meio_potencialmente_letal": "Meio potencialmente letal",
    "faixa_etaria": "Faixa etária",
    "fim_de_semana": "Fim de semana",
    "regiao_ocorrencia": "Região",
    "target_reincidencia": "Reincidência (alvo)",
    "encaminhamento_delegacia_mulher": "Encaminhamento à DEAM (alvo)",
}

SHORT_LABELS = {
    "idade_paciente": "Idade", "numero_envolvidos": "Nº agressores", "hora_ocorrencia_h": "Hora",
    "dia_semana_ocorrencia": "Dia semana", "mes_ocorrencia": "Mês", "uf_ocorrencia": "UF",
    "local_ocorrencia_cod": "Local", "reside_municipio_ocorrencia": "Mora no mun.",
    "raca_paciente": "Raça/cor", "escolaridade_paciente": "Escolaridade",
    "estado_civil_paciente": "Sit. conjugal", "autor_sexo": "Sexo agressor", "gestante": "Gestante",
    "possui_deficiencia": "Deficiência", "violencia_fisica": "V. física", "violencia_psicologica": "V. psicológica",
    "violencia_sexual": "V. sexual", "violencia_financeira": "V. financeira", "violencia_negligencia": "Negligência",
    "violencia_tortura": "Tortura", "meio_forca_corporal": "Força corporal", "meio_enforcamento": "Asfixia",
    "meio_objeto_contundente": "Obj. contundente", "meio_objeto_perfurante": "Obj. cortante",
    "meio_arma_fogo": "Arma de fogo", "meio_ameaca": "Ameaça", "autor_alcoolizado": "Álcool (agressor)",
    "autor_parceiro_ou_ex": "Parceiro/ex", "autor_familiar": "Familiar", "autor_conhecido": "Conhecido",
    "autor_desconhecido": "Desconhecido", "ocorrencia_antiga": "Fato antigo",
    "ocorreu_noite_madrugada": "Noite/madr.", "faixa_horaria": "Faixa horária", "local_residencia": "Residência",
    "n_tipos_violencia": "Nº tipos viol.", "n_meios_agressao": "Nº meios", "meio_potencialmente_letal": "Meio letal",
    "faixa_etaria": "Faixa etária", "fim_de_semana": "Fim de semana", "regiao_ocorrencia": "Região",
    "target_reincidencia": "Reincidência", "autor_parceiro_ou_ex_alvo": "Parceiro/ex",
    "encaminhamento_delegacia_mulher": "DEAM",
}

GROUPS = {
    "Vítima": ["idade_paciente", "faixa_etaria", "raca_paciente", "escolaridade_paciente",
               "estado_civil_paciente", "gestante", "possui_deficiencia"],
    "Quando e onde": ["hora_ocorrencia_h", "ocorreu_noite_madrugada", "faixa_horaria", "dia_semana_ocorrencia",
                      "fim_de_semana", "mes_ocorrencia", "ocorrencia_antiga", "uf_ocorrencia", "regiao_ocorrencia",
                      "local_ocorrencia_cod", "local_residencia", "reside_municipio_ocorrencia"],
    "Agressor": ["autor_sexo", "numero_envolvidos", "autor_alcoolizado", *VINCULO_COLS],
    "Tipos de violência": VIOLENCIA_COLS + ["n_tipos_violencia"],
    "Meios de agressão": MEIO_COLS + ["n_meios_agressao", "meio_potencialmente_letal"],
}
GROUP_OF = {col: g for g, cols in GROUPS.items() for col in cols}

_SIM_NAO = {"1": "Sim", "0": "Não"}
CODE_LABELS = {
    "dia_semana_ocorrencia": {"1": "Domingo", "2": "Segunda", "3": "Terça", "4": "Quarta",
                              "5": "Quinta", "6": "Sexta", "7": "Sábado"},
    "mes_ocorrencia": {"1": "Jan", "2": "Fev", "3": "Mar", "4": "Abr", "5": "Mai", "6": "Jun",
                       "7": "Jul", "8": "Ago", "9": "Set", "10": "Out", "11": "Nov", "12": "Dez"},
    "local_ocorrencia_cod": {"1": "Residência", "2": "Habitação coletiva", "3": "Escola",
                             "4": "Local de prática esportiva", "5": "Bar ou similar", "6": "Via pública",
                             "7": "Comércio/serviços", "8": "Indústria/construção", "9": "Outro"},
    "reside_municipio_ocorrencia": _SIM_NAO,
    "ocorreu_noite_madrugada": _SIM_NAO,
    "local_residencia": _SIM_NAO,
    "faixa_horaria": {"madrugada": "Madrugada (0–5h)", "manha": "Manhã (6–11h)",
                      "tarde": "Tarde (12–17h)", "noite": "Noite (18–23h)"},
    "raca_paciente": {"1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda", "5": "Indígena"},
    # Códigos originais do SINAN (CS_ESCOL_N)
    "escolaridade_paciente": {"0": "Analfabeta", "1": "1ª–4ª série incompleta", "2": "4ª série completa",
                              "3": "5ª–8ª série incompleta", "4": "Fundamental completo",
                              "5": "Médio incompleto", "6": "Médio completo",
                              "7": "Superior incompleto", "8": "Superior completo"},
    "estado_civil_paciente": {"1": "Solteira", "2": "Casada/união consensual", "3": "Viúva", "4": "Separada"},
    "autor_sexo": {"M": "Masculino", "F": "Feminino", "Ambos": "Ambos os sexos"},
    "numero_envolvidos": {"1": "Um", "2": "Dois ou mais"},
}


def label(col: str) -> str:
    return LABELS.get(col, col)


def short_label(col: str) -> str:
    return SHORT_LABELS.get(col, LABELS.get(col, col))


def code_label(col: str, code) -> str:
    code = normalize_code(code)
    if code == MISSING:
        return "Ignorado"
    return CODE_LABELS.get(col, {}).get(code, str(code))


In [ ]:
if "." not in sys.path:
    sys.path.insert(0, ".")
import features as F
importlib.reload(F)
print(f"{len(F.RAW_COLS)} campos brutos | {len(F.DERIVED_COLS)} derivadas | {len(F.ALL_CANDIDATES)} candidatas")

## 3. Dados e checagem de qualidade

In [ ]:
if IN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

t0 = time.time()
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"{len(df_raw):,} linhas × {df_raw.shape[1]} colunas ({time.time() - t0:.0f}s)")

esperadas = set(F.RAW_COLS) | set(F.TARGETS.values()) | {"ano"}
faltando = sorted(esperadas - set(df_raw.columns))
if faltando:
    print("ATENÇÃO — colunas ausentes (serão tratadas como 'ignorado'):", faltando)

# Os alvos devem ser 0/1/nulo. Se aparecer outro código, a query está desatualizada.
for alvo in F.TARGETS.values():
    vals = set(pd.to_numeric(df_raw[alvo], errors="coerce").dropna().unique())
    assert vals <= {0, 1}, f"{alvo} tem códigos inesperados: {vals} — confira a query"
    assert len(vals) == 2, f"{alvo} tem uma classe só: {vals} — confira a query (ex.: 'não' = 2)"

qual = df_raw.groupby("ano").agg(
    casos=("ano", "size"),
    **{f"{k} (n)": (v, "count") for k, v in [("reincid.", "target_reincidencia"), ("parceiro", "autor_parceiro_ou_ex"),
                                              ("DEAM", "encaminhamento_delegacia_mulher")]},
    **{f"{k} (taxa)": (v, "mean") for k, v in [("reincid.", "target_reincidencia"), ("parceiro", "autor_parceiro_ou_ex"),
                                                ("DEAM", "encaminhamento_delegacia_mulher")]},
    hora_ignorada=("hora_ocorrencia_h", lambda s: s.isna().mean()) if "hora_ocorrencia_h" in df_raw else ("ano", "size"),
)
display(qual.style.format({c: "{:.1%}" for c in qual.columns if "taxa" in c or c == "hora_ignorada"}
                          | {c: "{:,.0f}" for c in qual.columns if "(n)" in c or c == "casos"}))

In [ ]:
t0 = time.time()
Xp_all = F.prepare_features(df_raw)
ano_all = pd.to_numeric(df_raw["ano"], errors="coerce") if "ano" in df_raw else None
N_ROWS_CSV = len(df_raw)
# Libera memória: daqui em diante só precisamos dos alvos (o resto já está em Xp_all)
df_raw = df_raw[[c for c in F.TARGETS.values() if c in df_raw]].copy()
gc.collect()
print(f"Variáveis preparadas em {time.time() - t0:.0f}s | memória {Xp_all.memory_usage(deep=True).sum() / 1e9:.1f} GB")
resumo = pd.DataFrame({
    "tipo": [F.kind_of(c) for c in F.ALL_CANDIDATES],
    "derivada": [c in F.DERIVED for c in F.ALL_CANDIDATES],
    "% ignorado/nulo": [(Xp_all[c].isna() | (Xp_all[c] == F.MISSING)).mean() if F.kind_of(c) != "bin" else np.nan
                        for c in F.ALL_CANDIDATES],
    "níveis": [Xp_all[c].nunique() for c in F.ALL_CANDIDATES],
}, index=F.ALL_CANDIDATES)
display(resumo.style.format({"% ignorado/nulo": "{:.1%}"}, na_rep="–"))

## 4. Funções de treino

**Seleção de variáveis (RFE por permutação).** A seleção é feita separadamente para cada algoritmo:
1. Ajusta o modelo com todas as candidatas numa amostra do treino.
2. Mede a AUC na validação interna.
3. Calcula a importância por permutação de cada variável e remove a menos importante. Enquanto houver mais de 24 variáveis, remove duas por rodada.
4. Repete até sobrar uma.

Isso gera uma **curva de AUC × nº de variáveis**. Fica o menor *k* cuja AUC é maior ou igual à melhor AUC menos 1 erro-padrão (Hanley & McNeil). O conjunto de **teste não participa** da seleção nem da escolha do limiar.

**Modelos finais.**
* **Regressão logística:** imputação pela mediana e padronização nas numéricas, one-hot nas categóricas.
* **Gradient boosting** (`HistGradientBoostingClassifier`): categóricas nativas, `NaN` tratado nativamente e early stopping.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, classification_report,
                             confusion_matrix, log_loss, precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

ALGOS = {
    "lr": {"name": "Regressão Logística", "file_suffix": "lr"},
    "gbdt": {"name": "Gradient Boosting (GBDT)", "file_suffix": "tree"},
}
SCENARIO_EXCLUDE = {
    "target_reincidencia": {},
    "autor_parceiro_ou_ex": {c: "vínculo do agressor = o próprio alvo (vazamento)" for c in F.VINCULO_COLS},
    "encaminhamento_delegacia_mulher": {},
}
GLOBAL_EXCLUDED_INFO = {
    "encaminhamento_assistencia_social / atendimento_mulher": "decisão tomada no mesmo atendimento (vazamento)",
    "ano / data_ocorrencia": "usados só na validação temporal",
    "id/nome_municipio_ocorrencia": "alta cardinalidade (5 mil+ municípios); UF e região cobrem o território",
}
SUBGROUP_COLS = ["raca_paciente", "faixa_etaria", "regiao_ocorrencia"]

# Configuração da v2 (para o baseline nos mesmos dados)
V2_NUM = ["idade_paciente", "numero_envolvidos"]
V2_CAT = ["dia_semana_ocorrencia", "uf_ocorrencia", "raca_paciente", "escolaridade_paciente",
          "estado_civil_paciente", "autor_sexo"]
V2_BIN = ["ocorreu_noite_madrugada", "reside_municipio_ocorrencia", "local_residencia", "gestante",
          "possui_deficiencia", *F.VIOLENCIA_COLS, *F.MEIO_COLS, "autor_alcoolizado"]

T0 = time.time()


def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)


def hanley_se(auc, n_pos, n_neg):
    """Erro-padrão da AUC (Hanley & McNeil, 1982)."""
    q1, q2 = auc / (2 - auc), 2 * auc ** 2 / (1 + auc)
    var = (auc * (1 - auc) + (n_pos - 1) * (q1 - auc ** 2) + (n_neg - 1) * (q2 - auc ** 2)) / (n_pos * n_neg)
    return float(np.sqrt(max(var, 0.0)))


def subsample(X, y, n, seed):
    if n is None or len(y) <= n:
        return X, y
    Xs, _, ys, _ = train_test_split(X, y, train_size=n, stratify=y, random_state=seed)
    return Xs, ys


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        f = float(obj)
        return None if not np.isfinite(f) else round(f, 6)
    return obj


def build_pipeline(algo, feats, mode, with_prepare=False):
    """mode='select' (rápido, para a seleção) ou 'final'."""
    num = [f for f in feats if F.kind_of(f) == "num"]
    cat = [f for f in feats if F.kind_of(f) == "cat"]
    binc = [f for f in feats if F.kind_of(f) == "bin"]
    trs = []
    if algo == "lr":
        if num:
            trs.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num))
        if cat:
            trs.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat))  # esparso: menos memória
        if binc:
            trs.append(("bin", "passthrough", binc))
        model = LogisticRegression(C=1.0, max_iter=3000)
    else:
        if num:
            trs.append(("num", "passthrough", num))
        if cat:
            trs.append(("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                                              encoded_missing_value=np.nan), cat))
        if binc:
            trs.append(("bin", "passthrough", binc))
        mask = [False] * len(num) + [True] * len(cat) + [False] * len(binc)
        common = dict(categorical_features=mask if any(mask) else None, random_state=CFG["seed"],
                      min_samples_leaf=100, l2_regularization=1.0)
        if mode == "select":
            model = HistGradientBoostingClassifier(learning_rate=0.12, max_iter=CFG["gbdt_select_iter"],
                                                   max_leaf_nodes=31, early_stopping=False, **common)
        else:
            model = HistGradientBoostingClassifier(learning_rate=0.08, max_iter=CFG["gbdt_final_iter"],
                                                   max_leaf_nodes=48, early_stopping=True, validation_fraction=0.1,
                                                   n_iter_no_change=25, scoring="loss", **common)
    steps = [("prep", ColumnTransformer(trs, remainder="drop")), ("model", model)]
    if with_prepare:
        steps.insert(0, ("features", FunctionTransformer(F.prepare_features, validate=False)))
    return Pipeline(steps)


def build_v2_baseline(algo):
    """Reproduz o pré-processamento e os modelos do export_multi_models_v2.py."""
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), V2_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), V2_CAT),  # esparso só por memória
        ("bin", SimpleImputer(strategy="constant", fill_value=0), V2_BIN),
    ])
    if algo == "lr":
        model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    else:
        model = HistGradientBoostingClassifier(class_weight="balanced", max_iter=80, random_state=42)
    return Pipeline([("prep", prep), ("model", model)])


def v2_frame(Xp):
    """Na v2, noite/município/residência eram binárias com 'ignorado' = 0."""
    X = Xp[V2_NUM + V2_CAT + V2_BIN].copy()
    for c in ["ocorreu_noite_madrugada", "reside_municipio_ocorrencia", "local_residencia"]:
        X[c] = (X[c] == "1").astype(int)
    return X


def rfe_permutation(algo, candidates, Xf, yf, Xv, yv):
    remaining, steps = list(candidates), []
    n_pos, n_neg = int(yv.sum()), int((1 - yv).sum())
    while remaining:
        t = time.time()
        pipe = build_pipeline(algo, remaining, "select").fit(Xf[remaining], yf)
        auc = roc_auc_score(yv, pipe.predict_proba(Xv[remaining])[:, 1])
        step = {"k": len(remaining), "auc": auc, "se": hanley_se(auc, n_pos, n_neg), "features": list(remaining)}
        if len(remaining) > 1:
            pi = permutation_importance(pipe, Xv[remaining], yv, scoring="roc_auc",
                                        n_repeats=CFG["perm_repeats"], random_state=CFG["seed"], n_jobs=1)
            imp = dict(zip(remaining, pi.importances_mean))
            n_rm = 2 if len(remaining) > 24 else 1
            removed = sorted(remaining, key=lambda f: imp[f])[:n_rm]
            step["importance"] = {f: float(v) for f, v in imp.items()}
            step["removed"] = removed
            remaining = [f for f in remaining if f not in removed]
        else:
            remaining = []
        steps.append(step)
        print(f"      k={step['k']:2d}  AUC={auc:.4f}  remove {', '.join(step.get('removed', ['-']))} "
              f"({time.time() - t:.0f}s)", flush=True)
    return steps


def choose_subset(steps, tol=None):
    best = max(steps, key=lambda s: s["auc"])
    cut = best["auc"] - (best["se"] if tol is None else tol)
    chosen = min((s for s in steps if s["auc"] >= cut), key=lambda s: s["k"])
    return best, chosen, cut


def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(np.clip(thr[int(np.argmax(tpr - fpr))], 1e-4, 1 - 1e-4))


def curves(y, p):
    grid = np.linspace(0, 1, 101)
    fpr, tpr, _ = roc_curve(y, p)
    prec, rec, _ = precision_recall_curve(y, p)
    order = np.argsort(rec)
    prob_true, prob_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
    bins = np.linspace(0, 1, 101)
    return {
        "roc": {"fpr": grid, "tpr": np.interp(grid, fpr, tpr)},
        "pr": {"recall": grid, "precision": np.interp(grid, rec[order], prec[order])},
        "calibration": {"mean_predicted": prob_pred, "observed": prob_true},
        "score_hist": {"bin_edges": bins, "pos": np.histogram(p[y == 1], bins=bins)[0],
                       "neg": np.histogram(p[y == 0], bins=bins)[0]},
    }


def eval_at(y, p, thr):
    yhat = (p >= thr).astype(int)
    auc = roc_auc_score(y, p)
    se = hanley_se(auc, int(y.sum()), int((1 - y).sum()))
    return {"threshold": thr, "roc_auc": auc, "roc_auc_ci95": [auc - 1.96 * se, auc + 1.96 * se],
            "average_precision": average_precision_score(y, p), "brier": brier_score_loss(y, p),
            "log_loss": log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)),
            "report": classification_report(y, yhat, output_dict=True, zero_division=0),
            "confusion_matrix": confusion_matrix(y, yhat, labels=[0, 1])}


def subgroup_metrics(Xp, y, p, thr):
    out, yhat, yv = {}, (p >= thr).astype(int), np.asarray(y)
    for col in SUBGROUP_COLS:
        rows = []
        for g in Xp[col].unique():
            m = (Xp[col] == g).to_numpy()
            yy, pp, hh = yv[m], p[m], yhat[m]
            if m.sum() < 300 or yy.min() == yy.max():
                continue
            tp, fp = int(((hh == 1) & (yy == 1)).sum()), int(((hh == 1) & (yy == 0)).sum())
            fn, tn = int(((hh == 0) & (yy == 1)).sum()), int(((hh == 0) & (yy == 0)).sum())
            rows.append({"group": str(g), "n": int(m.sum()), "prevalence": float(yy.mean()),
                         "mean_pred": float(pp.mean()), "auc": float(roc_auc_score(yy, pp)),
                         "recall": tp / max(tp + fn, 1), "fpr": fp / max(fp + tn, 1),
                         "precision": tp / max(tp + fp, 1)})
        out[col] = sorted(rows, key=lambda r: -r["n"])
    return out


def lr_coefficients(pipe):
    prep, model = pipe.named_steps["prep"], pipe.named_steps["model"]
    cat_cols = [c for c in F.ALL_CANDIDATES if F.kind_of(c) == "cat"]
    rows = []
    for name, c in zip(prep.get_feature_names_out(), model.coef_.ravel()):
        block, _, rest = name.partition("__")
        if block == "cat":
            col = max((f for f in cat_cols if rest.startswith(f + "_")), key=len, default=rest)
            cat = rest[len(col) + 1:]
        else:
            col, cat = rest, None
        rows.append({"column": col, "category": cat, "block": block, "coef": float(c), "odds_ratio": float(np.exp(c))})
    return {"intercept": float(model.intercept_[0]), "terms": rows}


def cramers_v(x, y):
    ct = pd.crosstab(np.asarray(x), np.asarray(y))
    if min(ct.shape) < 2:
        return 0.0
    n = ct.to_numpy().sum()
    exp = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / n
    chi2 = ((ct.to_numpy() - exp) ** 2 / exp).sum()
    return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))


def category_rates(get_col, y, cols):
    """get_col(c) -> array com os valores da coluna no treino."""
    out = {}
    for c in cols:
        v = pd.Series(get_col(c))
        if F.kind_of(c) == "num":
            v = v.map(lambda x: F.MISSING if pd.isna(x) else str(int(x)))
        g = pd.DataFrame({"v": v.astype(str).to_numpy(), "y": np.asarray(y)}).groupby("v")["y"].agg(["size", "mean"])
        out[c] = [{"code": k, "n": int(r["size"]), "rate": float(r["mean"])} for k, r in g.iterrows()]
    return out


print("Funções prontas.")

## 5. Treino

A célula abaixo roda os cenários de `CENARIOS`. Ao fim de cada cenário, os modelos e as métricas daquele cenário já ficam salvos em `PASTA_SAIDA`. Se a sessão cair, rode de novo só com os cenários que faltaram, e os resultados são mesclados.

A divisão treino/teste é **80/20 estratificada** com `random_state=42`. Dentro do treino, **15%** vão para a validação interna, usada na seleção e na escolha do limiar.

In [ ]:
out = Path(PASTA_SAIDA)
out.mkdir(parents=True, exist_ok=True)
metrics_path, samples_path = out / "metrics.json", out / "test_samples.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.exists() else {"scenarios": {}}
samples_out = json.loads(samples_path.read_text(encoding="utf-8")) if samples_path.exists() else {}

anos = ano_all.dropna().astype(int) if ano_all is not None else pd.Series(dtype=int)
metrics["meta"] = {
    "version": "v3",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(), "sklearn_version": sklearn.__version__,
    "pandas_version": pd.__version__, "numpy_version": np.__version__,
    "n_rows_csv": int(N_ROWS_CSV), "years": [int(anos.min()), int(anos.max())] if len(anos) else None,
    "rows_per_year": {int(k): int(v) for k, v in anos.value_counts().sort_index().items()},
    "config": CFG, "modo_rapido": MODO_RAPIDO,
    "selection_method": ("Eliminação recursiva por importância de permutação (AUC) numa validação interna; "
                         "escolhe o menor subconjunto com AUC >= melhor AUC − 1 erro-padrão."),
    "global_excluded": GLOBAL_EXCLUDED_INFO,
    "derived": {k: v["sources"] for k, v in F.DERIVED.items()},
}

def take(rows, cols=None):
    """Copia só as linhas (posições) e colunas pedidas de Xp_all — evita cópias inteiras na memória."""
    if cols is None:
        return Xp_all.iloc[rows]
    return Xp_all.iloc[rows, [Xp_all.columns.get_loc(c) for c in cols]]


def full_pipeline(inner, X_raw_sample):
    """Acopla o preparo de variáveis (features.prepare_features) ao pipeline já ajustado em dados preparados.
    O app passa a linha crua e o pipeline faz tipagem, normalização e derivadas."""
    ft = FunctionTransformer(F.prepare_features, validate=False).fit(X_raw_sample)
    return Pipeline([("features", ft), ("prep", inner.named_steps["prep"]), ("model", inner.named_steps["model"])])


for label, target in F.TARGETS.items():
    if target not in CENARIOS:
        continue
    log(f"=== {label} ({target}) ===")
    idx_all = np.flatnonzero(df_raw[target].notna().to_numpy())
    y_all = pd.to_numeric(df_raw[target].iloc[idx_all], errors="coerce").astype(int).to_numpy()
    ano = ano_all.to_numpy()[idx_all] if ano_all is not None else None

    excl = dict(SCENARIO_EXCLUDE[target]) | {target: "é o alvo"}
    candidates = [c for c in F.ALL_CANDIDATES if c not in excl]

    # Divisão por índices (o embaralhamento depende só de n, y e random_state — igual a dividir o DataFrame)
    i_train, i_test, y_train, y_test = train_test_split(idx_all, y_all, test_size=0.20, stratify=y_all,
                                                        random_state=42)
    i_tr, i_val, y_tr, y_val = train_test_split(i_train, y_train, test_size=0.15, stratify=y_train,
                                                random_state=CFG["seed"])
    i_tr, y_tr = subsample(i_tr, y_tr, CFG["max_train"], CFG["seed"])
    X_tr, X_val, X_test = take(i_tr), take(i_val), take(i_test)
    Xs_fit, ys_fit = subsample(X_tr, y_tr, CFG["sel_fit"], CFG["seed"])
    Xs_val, ys_val = subsample(X_val, y_val, CFG["sel_val"], CFG["seed"])
    log(f"treino {len(i_train):,} (ajuste {len(i_tr):,} | validação {len(i_val):,}) | teste {len(i_test):,} | "
        f"{len(candidates)} candidatas | prevalência {y_all.mean():.1%}")

    # ---- Descritivos (somente treino) ----
    def col_train(c):
        return Xp_all[c].to_numpy()[i_train]

    num_bin = [c for c in candidates if F.kind_of(c) in ("num", "bin")]
    corr = pd.concat([take(i_train, num_bin).reset_index(drop=True), pd.Series(y_train, name=target)],
                     axis=1).corr(method="pearson")
    assoc = [{"feature": c, "measure": "cramers_v", "value": cramers_v(col_train(c), y_train)} if F.kind_of(c) == "cat"
             else {"feature": c, "measure": "pearson", "value": float(corr.loc[c, target])} for c in candidates]
    rate_cols = list(dict.fromkeys(c for c in [c for c in candidates if F.kind_of(c) in ("cat", "bin")]
                                   + ["numero_envolvidos"] if c in candidates))
    reference = {}
    for c in F.RAW_COLS:
        v = pd.Series(col_train(c))
        reference[c] = float(v.median()) if F.kind_of(c) == "num" else v.mode().iloc[0]
    scen = {
        "slug": target, "n_total": int(len(y_all)), "n_train": int(len(i_train)), "n_test": int(len(i_test)),
        "n_fit": int(len(i_tr)), "n_val": int(len(i_val)),
        "prevalence_train": float(y_train.mean()), "prevalence_test": float(y_test.mean()),
        "years": [int(ano.min()), int(ano.max())] if ano is not None else None,
        "rate_by_year": ({int(k): float(v) for k, v in pd.Series(y_all).groupby(ano).mean().items()}
                         if ano is not None else {}),
        "candidates": candidates, "excluded": excl,
        "full_correlation_matrix": {"columns": corr.columns.tolist(), "index": corr.index.tolist(),
                                    "data": corr.round(3).to_numpy()},
        "target_association": sorted(assoc, key=lambda r: -abs(r["value"])),
        "category_rates": category_rates(col_train, y_train, rate_cols),
        "reference_profile": reference,
        "categories": {c: sorted(pd.unique(col_train(c)).astype(str).tolist())
                       for c in F.ALL_CANDIDATES if F.kind_of(c) == "cat"},
        "models": {},
    }

    test_preds = {}
    for algo, meta in ALGOS.items():
        log(f"  [{meta['name']}] seleção de variáveis")
        steps = rfe_permutation(algo, candidates, Xs_fit, ys_fit, Xs_val, ys_val)
        best, chosen, cut = choose_subset(steps, CFG["tol"])
        feats = [c for c in candidates if c in chosen["features"]]
        log(f"  [{meta['name']}] melhor k={best['k']} (AUC {best['auc']:.4f}) → escolhido k={chosen['k']} "
            f"(AUC {chosen['auc']:.4f}, corte {cut:.4f})")
        print("      ", feats)

        t = time.time()
        inner = build_pipeline(algo, feats, "final").fit(X_tr[feats], y_tr)
        pipe = full_pipeline(inner, X_tr[F.RAW_COLS].head(200))
        log(f"  [{meta['name']}] ajuste final ({time.time() - t:.0f}s"
            + (f", {inner.named_steps['model'].n_iter_} iterações)" if algo == "gbdt" else ")"))

        # Consistência: o pipeline salvo (linha crua) tem de prever exatamente o mesmo que o modelo ajustado
        chk = X_test.iloc[:3000]
        diff = np.abs(pipe.predict_proba(chk[F.RAW_COLS])[:, 1] - inner.predict_proba(chk[feats])[:, 1]).max()
        assert diff < 1e-9, f"pipeline completo diverge do modelo ajustado (máx. diferença {diff})"

        thr = youden_threshold(y_val, inner.predict_proba(X_val[feats])[:, 1])
        p_test = inner.predict_proba(X_test[feats])[:, 1]
        test_preds[algo] = p_test
        ev = eval_at(y_test, p_test, thr)
        log(f"  [{meta['name']}] TESTE  AUC {ev['roc_auc']:.4f} | AP {ev['average_precision']:.4f} | limiar {thr:.3f} | "
            f"recall {ev['report']['1']['recall']:.3f} | precisão {ev['report']['1']['precision']:.3f}")

        Xi, yi = subsample(X_val, y_val, 50_000, CFG["seed"])
        pi = permutation_importance(inner, Xi[feats], yi, scoring="roc_auc",
                                    n_repeats=max(CFG["perm_repeats"], 5), random_state=CFG["seed"], n_jobs=1)
        info = {
            "name": meta["name"], "file": f"model_{target}_{meta['file_suffix']}.joblib",
            "selected_features": feats, "raw_inputs": F.raw_inputs_for(feats),
            "selection": {"steps": steps, "best_k": best["k"], "best_auc": best["auc"], "chosen_k": chosen["k"],
                          "chosen_auc": chosen["auc"], "cutoff": cut, "n_fit": len(Xs_fit), "n_val": len(Xs_val)},
            "threshold": thr, "test": ev,
            "curves": curves(y_test, p_test),
            "importance": sorted([{"feature": f, "mean": float(m), "std": float(s)} for f, m, s in
                                  zip(feats, pi.importances_mean, pi.importances_std)], key=lambda r: -r["mean"]),
            "subgroups": subgroup_metrics(X_test, y_test, p_test, thr),
        }
        if algo == "gbdt":
            info["n_iter"] = int(inner.named_steps["model"].n_iter_)
        else:
            info["coefficients"] = lr_coefficients(pipe)
        joblib.dump(pipe, out / info["file"], compress=3)
        del inner, chk, Xi
        gc.collect()

        # ---- Validação temporal ----
        if ano is not None:
            past, fut = ano <= CFG["temporal_corte"], ano > CFG["temporal_corte"]
            if past.sum() > 1000 and fut.sum() > 1000:
                i_pa, y_pa = subsample(idx_all[past], y_all[past], CFG["temporal_max_train"], CFG["seed"])
                pt = build_pipeline(algo, feats, "final").fit(take(i_pa, feats), y_pa)
                yf = y_all[fut]
                pf = pt.predict_proba(take(idx_all[fut], feats))[:, 1]
                info["temporal"] = {"train_years": f"{int(ano[past].min())}–{CFG['temporal_corte']}",
                                    "test_years": f"{CFG['temporal_corte'] + 1}–{int(ano[fut].max())}",
                                    "n_train": int(len(y_pa)), "n_test": int(len(yf)),
                                    "auc": float(roc_auc_score(yf, pf)),
                                    "average_precision": float(average_precision_score(yf, pf)),
                                    "prevalence_train": float(y_pa.mean()), "prevalence_test": float(yf.mean()),
                                    "mean_pred_test": float(pf.mean())}
                log(f"  [{meta['name']}] temporal AUC {info['temporal']['auc']:.4f} "
                    f"(taxa real {yf.mean():.1%} × prevista {pf.mean():.1%})")
                del pt, pf
                gc.collect()

        # ---- Baseline: configuração v2, mesmas linhas de ajuste e mesmo teste ----
        t = time.time()
        pb = build_v2_baseline(algo).fit(v2_frame(X_tr), y_tr).predict_proba(v2_frame(X_test))[:, 1]
        evb = eval_at(y_test, pb, 0.5)
        info["baseline_v2"] = {"roc_auc": evb["roc_auc"], "average_precision": evb["average_precision"],
                               "brier": evb["brier"], "recall": evb["report"]["1"]["recall"],
                               "precision": evb["report"]["1"]["precision"], "f1": evb["report"]["1"]["f1-score"],
                               "confusion_matrix": evb["confusion_matrix"], "threshold": 0.5,
                               "n_features": len(V2_NUM + V2_CAT + V2_BIN),
                               "note": ("config. v2 (26 variáveis, class_weight='balanced', limiar 0,5) retreinada "
                                        "nas mesmas linhas de ajuste e avaliada no mesmo teste")}
        log(f"  [{meta['name']}] baseline v2 AUC {evb['roc_auc']:.4f} ({time.time() - t:.0f}s)")
        del pb
        gc.collect()
        scen["models"][algo] = info

    scen["model_agreement"] = {"spearman": float(pd.Series(test_preds["lr"]).corr(pd.Series(test_preds["gbdt"]),
                                                                                  method="spearman"))}

    # ---- Casos de teste para o simulador ----
    tdf = X_test[F.RAW_COLS].copy()
    tdf["target_real"] = y_test
    for algo in ALGOS:
        tdf[f"_p_{algo}"] = test_preds[algo]
    amostra = pd.concat([tdf[tdf.target_real == 1].sample(10, random_state=CFG["seed"]),
                         tdf[tdf.target_real == 0].sample(10, random_state=CFG["seed"])]).sample(frac=1, random_state=CFG["seed"])
    recs = []
    for idx, r in amostra.iterrows():
        rec = {}
        for c in F.RAW_COLS:
            v = r[c]
            rec[c] = None if (v is None or (isinstance(v, float) and np.isnan(v))) else (
                str(v) if F.kind_of(c) == "cat" else float(v) if F.kind_of(c) == "num" else int(v))
        rec.update(target_real=int(r["target_real"]), row_id=int(idx),
                   p_treino={a: float(r[f"_p_{a}"]) for a in ALGOS})
        if ano_all is not None:
            rec["ano"] = int(ano_all.loc[idx])
        recs.append(rec)
    samples_out[label] = recs
    metrics["scenarios"][label] = scen

    # Salva a cada cenário (permite retomar)
    metrics_path.write_text(json.dumps(to_jsonable(metrics), ensure_ascii=False), encoding="utf-8")
    samples_path.write_text(json.dumps(to_jsonable(samples_out), ensure_ascii=False, indent=1), encoding="utf-8")
    log(f"  cenário salvo em {out}/")
    del X_tr, X_val, X_test, Xs_fit, Xs_val, tdf, amostra, test_preds
    gc.collect()

log("Treino concluído.")

## 6. Resultados

In [ ]:
linhas = []
for label, scen in metrics["scenarios"].items():
    for algo, m in scen["models"].items():
        b = m.get("baseline_v2", {})
        linhas.append({
            "cenário": label, "modelo": m["name"],
            "nº variáveis": len(m["selected_features"]), "AUC v3": m["test"]["roc_auc"],
            "AUC v2 (mesmos dados)": b.get("roc_auc"), "Δ AUC": m["test"]["roc_auc"] - b.get("roc_auc", np.nan),
            "AP v3": m["test"]["average_precision"], "AP v2": b.get("average_precision"),
            "recall v3": m["test"]["report"]["1"]["recall"], "precisão v3": m["test"]["report"]["1"]["precision"],
            "limiar": m["threshold"], "AUC temporal": m.get("temporal", {}).get("auc"),
        })
tab = pd.DataFrame(linhas)
display(tab.style.format({c: "{:.3f}" for c in tab.columns if tab[c].dtype == float}))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(metrics["scenarios"]), figsize=(5.2 * len(metrics["scenarios"]), 3.8), squeeze=False)
for ax, (label, scen) in zip(axes[0], metrics["scenarios"].items()):
    for algo, color in [("lr", "#2a78d6"), ("gbdt", "#d6582a")]:
        m = scen["models"][algo]
        st = m["selection"]["steps"]
        ax.plot([s["k"] for s in st], [s["auc"] for s in st], marker="o", ms=3, lw=1.6, color=color, label=m["name"])
        ax.axvline(m["selection"]["chosen_k"], color=color, ls=":", lw=1)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("nº de variáveis")
    ax.invert_xaxis()
    ax.grid(alpha=.25)
axes[0][0].set_ylabel("AUC na validação interna")
axes[0][0].legend(fontsize=8, frameon=False)
fig.suptitle("Curvas de seleção (linha pontilhada = subconjunto escolhido)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
sel = pd.DataFrame({f"{label.split(' (')[0]} · {m['name'].split(' (')[0]}":
                    pd.Series(1, index=m["selected_features"])
                    for label, scen in metrics["scenarios"].items() for m in scen["models"].values()}).notna()
sel = sel.loc[[c for c in F.ALL_CANDIDATES if c in sel.index]]
sel.index = [F.label(c) for c in sel.index]
display(sel.replace({True: "●", False: ""}))

## 7. Exportar

Gera o `modelos_v3.zip` com os 6 `.joblib`, o `metrics.json`, o `test_samples.json`, o `features.py` e um `requirements.txt` com a **mesma versão do scikit-learn usada aqui**. Sem isso, o Streamlit Cloud pode não conseguir carregar os modelos.

No repositório do app: extraia o zip na raiz, substituindo os arquivos antigos, e faça o commit.

In [ ]:
shutil.copy("features.py", out / "features.py")
(out / "requirements.txt").write_text(
    "streamlit>=1.40\n"
    f"scikit-learn=={sklearn.__version__}\n"
    f"pandas>={pd.__version__.split('.')[0]}.{pd.__version__.split('.')[1]}\n"
    f"numpy>={np.__version__.split('.')[0]}.0,<{int(np.__version__.split('.')[0]) + 1}\n"
    "joblib>=1.3\n"
    "plotly>=5.24\n", encoding="utf-8")

zip_path = Path("modelos_v3.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        z.write(f, arcname=f.name)
print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB):", ", ".join(sorted(f.name for f in out.iterdir())))

if IN_COLAB:
    if COPIAR_PARA_DRIVE and USAR_DRIVE:
        Path(PASTA_DRIVE_SAIDA).mkdir(parents=True, exist_ok=True)
        shutil.copy(zip_path, Path(PASTA_DRIVE_SAIDA) / zip_path.name)
        print("Cópia salva em", PASTA_DRIVE_SAIDA)
    from google.colab import files
    files.download(str(zip_path))